# ZODB explorer
Step-wise walk through the object tree of a Zope `Data.fs` to find object ids that are **not listed by the Zope UI** (`objectIds()`) but are **still reachable/persisted** in the database.

The storage is opened **read-only**, so it is safe even while Zope is running.

In [1]:
import os, pprint
import ZODB
from ZODB.FileStorage import FileStorage
from ZODB.utils import u64, p64
from persistent import Persistent

wd = '/home/zope/instance/zms5_dev/var'  # adapt to your Zope instance
try:
    db.close()
except NameError:
    pass
db = ZODB.DB(FileStorage(os.path.join(wd, 'Data.fs'), read_only=True))
conn = db.open()
root = conn.root()
app = root['Application']
print(list(root.keys()), app)

['Application'] <Application at >


## Helpers
- `ui_ids(obj)`: ids that the Zope UI shows (`objectIds()`).
- `stored_children(obj)`: every persistent sub-object found in the raw state (`__dict__`, plus BTree-folder `_tree`), i.e. what really lives in the database.
- `hidden(obj)`: stored but not shown in the UI.

In [18]:
def ui_ids(obj):
    try:
        return set(obj.objectIds())
    except Exception:
        return set()

def raw_state(obj):
    """Raw attribute dict without triggering acquisition."""
    obj._p_activate()
    return dict(getattr(obj, '__dict__', {}))

def stored_children(obj):
    """{key: child} for persistent children in raw state and BTree containers."""
    res = {}
    base = getattr(obj, 'aq_base', obj)
    for k, v in raw_state(base).items():
        if isinstance(v, Persistent):
            res[k] = v
    tree = getattr(base, '_tree', None)  # BTreeFolder2
    if tree is not None:
        for k in tree.keys():
            res[k] = tree[k]
    return res

def hidden(obj):
    shown = ui_ids(obj)
    return {k: v for k, v in stored_children(obj).items() if k not in shown}

def describe(obj):
    return '%s oid=%s %s.%s' % (getattr(obj, 'id', None) or getattr(obj, 'getId', lambda: '')(),
        hex(u64(obj._p_oid)), type(obj).__module__, type(obj).__name__)

def show(obj):
    """One step: print children of obj, marking those hidden from the UI."""
    shown = ui_ids(obj)
    print('==', describe(obj))
    for k, v in sorted(stored_children(obj).items()):
        flag = 'UI    ' if k in shown else 'HIDDEN'
        print('  [%s] %-30s %s' % (flag, k, describe(v)))
    scalars = {k: v for k, v in raw_state(getattr(obj, 'aq_base', obj)).items() if not isinstance(v, Persistent)}
    print('  non-persistent attributes:', sorted(scalars))

## Step 1: the Application root

In [19]:
show(app)

== Zope oid=0x1 OFS.Application.Application
  [HIDDEN] __allow_groups__               acl_users oid=0x2 OFS.userfolder.UserFolder
  [HIDDEN] __error_log__                  error_log oid=0x8 Products.SiteErrorLog.SiteErrorLog.SiteErrorLog
  [UI    ] acl_users                      acl_users oid=0x2 OFS.userfolder.UserFolder
  [UI    ] browser_id_manager             browser_id_manager oid=0x5 Products.Sessions.BrowserIdManager.BrowserIdManager
  [UI    ] desy                           desy oid=0x1c772 OFS.Folder.Folder
  [UI    ] error_log                      error_log oid=0x8 Products.SiteErrorLog.SiteErrorLog.SiteErrorLog
  [UI    ] favicon.ico                    favicon.ico oid=0x19704 OFS.Image.Image
  [UI    ] fonts                          fonts oid=0x41eb2 Products.CMFCore.DirectoryView.DirectoryView
  [UI    ] headCache                      headCache oid=0x41eb3 Products.PythonScripts.PythonScript.PythonScript
  [UI    ] index_html                     index_html oid=0xb Products.

## Step 2: descend one level at a time
Set `path` to a list of ids; `walk` follows them via the stored children (so hidden objects can be entered as well).

In [4]:
def walk(path, start=None):
    obj = start if start is not None else app
    for p in path:
        obj = stored_children(obj)[p]
    return obj

path = ['unibe','subclient01','content']  # e.g. ['myzmsx']
node = walk(path)
show(node)

== content oid=0x1ae2a Products.zms.zms.ZMS
  [UI    ] 1752745038.6409214             1752745038.6409214 oid=0x1b1c0 Products.zms._zmsattributecontainer.ZMSAttributeContainer
  [UI    ] acl_sequence                   acl_sequence oid=0x1b1c5 Products.zms._sequence.Sequence
  [UI    ] catalog_ger                    catalog_ger oid=0x1b30c Products.ZCatalog.ZCatalog.ZCatalog
  [UI    ] e12                            e12 oid=0x1b1c6 Products.zms.zmscustom.ZMSCustom
  [UI    ] e2                             e2 oid=0x1b1c2 Products.zms.zmscustom.ZMSCustom
  [UI    ] e22                            e22 oid=0x1b1c7 Products.zms.zmscustom.ZMSCustom
  [UI    ] e34                            e34 oid=0x1b1c8 Products.zms.zmscustom.ZMSCustom
  [UI    ] e4                             e4 oid=0x1b1c3 Products.zms.zmscustom.ZMSCustom
  [UI    ] e44                            e44 oid=0x1b562 Products.zms.zmscustom.ZMSCustom
  [UI    ] e45                            e45 oid=0x1b1c9 Products.zms.zmscustom

## Step 2b: object structure (all properties and child-objects)
`props(obj)` prints all non-persistent attributes from the raw state (strings, numbers, dicts, lists, ...), with type and a shortened value. Use `full=True` for untruncated values, `only='substr'` to filter names.

In [52]:
def props(obj, full=False, only=None, maxlen=120):
    """Print all data (non-persistent) attributes of the raw object state."""
    base = getattr(obj, 'aq_base', obj)
    print('==', describe(base))
    for k, v in sorted(raw_state(base).items()):
        if isinstance(v, Persistent) or (only and only not in k):
            continue
        r = pprint.pformat(v, width=100, compact=True)
        if not full and len(r) > maxlen:
            r = r[:maxlen] + ' ... (%d chars)' % len(r)
        print('  %-28s %-14s %s' % (k, type(v).__name__, r))

props(node)

== content oid=0x1ae2a Products.zms.zms.ZMS
  _Access_Transient_Objects_Permission list           ['Manager', 'ZMSAdministrator']
  _Access_arbitrary_user_session_data_Permission list           ['Manager', 'ZMSAdministrator']
  _Access_contents_information_Permission list           ['Manager', 'ZMSAdministrator', 'ZMSEditor', 'ZMSAuthor', 'ZMSSubscriber', 'ZMSUserAdministrator']
  _Access_inactive_portal_content_Permission list           ['Manager', 'ZMSAdministrator']
  _Access_session_data_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_Accelerated_HTTP_Cache_Managers_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_BTreeFolder2s_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_Browser_Id_Manager_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_CMF_Caching_Policy_Managers_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_CMF_Core_Tools_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_Conte

# Step 2c: object properties

In [51]:
print("Properties of the node: %s /  %s /  %s"%(node.id, node._p_oid, node._p_serial))
print("##########")
for k in node.__dict__.keys():
    # print only properties and not objects
    if not hasattr(node.__dict__[k], '__dict__'):
        print(k, node.__dict__[k], type(node.__dict__[k]))

Properties of the node: content /  b'\x00\x00\x00\x00\x00\x01\xae*' /  b'\x04\x0c\xc5\x11W\xc4\xaa3'
##########
id content <class 'str'>
__ac_local_roles__ {'admin': ['Owner']} <class 'dict'>
_objects ({'id': 'trashcan', 'meta_type': 'ZMSTrashcan'}, {'id': 'sys_conf', 'meta_type': 'ZMSSysConf'}, {'id': 'metaobj_manager', 'meta_type': 'ZMSMetamodelProvider'}, {'id': 'metacmd_manager', 'meta_type': 'ZMSMetacmdProvider'}, {'id': 'format_manager', 'meta_type': 'ZMSFormatProvider'}, {'id': 'zmsindex', 'meta_type': 'ZMSIndex'}, {'id': 'manage_tab_ZMSIndexZCatalog', 'meta_type': 'External Method'}, {'id': 'manage_tab_statistics', 'meta_type': 'Page Template'}, {'id': '1752745038.6409214', 'meta_type': 'ZMSAttributeContainer'}, {'id': 'zcatalog_adapter', 'meta_type': 'ZMSZCatalogAdapter'}, {'id': 'e2', 'meta_type': 'ZMSCustom'}, {'id': 'e4', 'meta_type': 'ZMSCustom'}, {'id': 'e5', 'meta_type': 'ZMSCustom'}, {'id': 'acl_sequence', 'meta_type': 'Sequence'}, {'id': 'e12', 'meta_type': 'ZMSCustom'

## Step 3: scan the whole tree for hidden ids
Optionally filter by `wanted` ids (set empty to list all hidden objects).

In [ ]:
wanted = set()  # e.g. {'content', 'manage_main'}
max_depth = 12

def scan(obj, path=(), seen=None, depth=0):
    seen = seen if seen is not None else set()
    if obj._p_oid in seen or depth > max_depth:
        return
    seen.add(obj._p_oid)
    for k, v in hidden(obj).items():
        if not wanted or k in wanted:
            print('/'.join(path) or '/', '->', k, describe(v))
    for k, v in stored_children(obj).items():
        scan(v, path + (k,), seen, depth + 1)

scan(app)

## Step 4: database-level view
Count all current objects in the storage and those **not reachable** from the root (garbage that `pack` would remove). Reachable-but-hidden objects found in step 3 survive packing.

In [22]:
from ZODB.serialize import referencesf
from collections import Counter

storage = db.storage
all_oids, refs = set(), {}
oid = None
while True:
    oid, tid, data, next_oid = storage.record_iternext(oid)
    all_oids.add(oid)
    refs[oid] = referencesf(data)
    if next_oid is None:
        break
    oid = next_oid

reach, todo = set(), [p64(0)]
while todo:
    o = todo.pop()
    if o in reach or o not in refs:
        continue
    reach.add(o)
    todo.extend(refs[o])
print('objects:', len(all_oids), 'reachable:', len(reach), 'unreachable:', len(all_oids - reach))

objects: 33020 reachable: 30626 unreachable: 2394


In [ ]:
# Class histogram of unreachable objects (candidates for the next pack)
from ZODB.utils import get_pickle_metadata
c = Counter()
for o in all_oids - reach:
    data, _ = storage.load(o)
    c['%s.%s' % get_pickle_metadata(data)] += 1
pprint.pprint(c.most_common(20))

## ZODB Property Editor
Changes a data property in the raw object state. **This needs a writable storage**, so:
1. **Stop Zope first** (a running instance holds `Data.fs.lock`; opening writable fails otherwise), and make a backup of `Data.fs`.
2. Run `reopen(read_only=False)`.
3. `set_prop(obj, name, value)` shows old/new value and changes it in memory only (dry run until `commit=True`).
4. `set_prop(..., commit=True)` writes the transaction. `abort()` discards pending changes. Use `reopen(read_only=True)` to switch back.

`value` is a Python literal string (`"'text'"`, `"42"`, `"{'a': 1}"`, `"None"`) or any Python object.

In [6]:
import ast, transaction

def reopen(read_only=True):
    """Close and reopen the DB; refreshes the globals db, conn, root, app, node."""
    global db, conn, root, app, node
    try:
        transaction.abort()
        conn.close()
        db.close()
    except Exception:
        pass
    db = ZODB.DB(FileStorage(os.path.join(wd, 'Data.fs'), read_only=read_only))
    conn = db.open()
    root = conn.root()
    app = root['Application']
    node = walk(path)
    print('opened', 'READ-ONLY' if read_only else 'WRITABLE', '- node:', describe(node))

def abort():
    transaction.abort()
    print('aborted pending changes')

def set_prop(obj, name, value, commit=False):
    """Set data property `name` on obj. Strings are parsed as Python literals."""
    if db.storage.isReadOnly():
        raise RuntimeError('storage is read-only; run reopen(read_only=False) (Zope must be stopped)')
    base = getattr(obj, 'aq_base', obj)
    state = raw_state(base)
    if name not in state:
        print('note: property %r does not exist yet and will be created' % name)
    if isinstance(state.get(name), Persistent):
        raise ValueError('%r is a persistent sub-object, not a data property' % name)
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            raise ValueError('value is not a valid Python literal: %r (quote strings, e.g. "\'text\'")' % value)
    print('old:', pprint.pformat(state.get(name), width=100))
    print('new:', pprint.pformat(value, width=100))
    setattr(base, name, value)
    base._p_changed = True
    if commit:
        transaction.get().note('zodb_explorer: set %s on %s' % (name, describe(base)))
        transaction.commit()
        print('COMMITTED')
    else:
        print('pending (dry run) - call with commit=True or abort()')

# Example (writable storage required):
# reopen(read_only=False)
# set_prop(node, 'title', "'New title'")                # dry run
# set_prop(node, 'title', "'New title'", commit=True)   # write
# props(node, only='title')

In [17]:
# Reopen the ZODB connection in read-write mode (maybe delete Data.fs.lock manually)
reopen(read_only=False)

opened WRITABLE - node: content oid=0x1ae2a Products.zms.zms.ZMS


In [ ]:
# Set the 'version_live_id' property of the current ZODB node and commit the change
set_prop(node, 'version_live_id', "'1752745038.6409001'", commit=True)

old: '1752745038.6409000'
new: '1752745038.6409001'
COMMITTED


In [ ]:
# Show the current properties of the current ZODB node
props(node)

== content oid=0x1ae2a Products.zms.zms.ZMS
  _Access_Transient_Objects_Permission list           ['Manager', 'ZMSAdministrator']
  _Access_arbitrary_user_session_data_Permission list           ['Manager', 'ZMSAdministrator']
  _Access_contents_information_Permission list           ['Manager', 'ZMSAdministrator', 'ZMSEditor', 'ZMSAuthor', 'ZMSSubscriber', 'ZMSUserAdministrator']
  _Access_inactive_portal_content_Permission list           ['Manager', 'ZMSAdministrator']
  _Access_session_data_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_Accelerated_HTTP_Cache_Managers_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_BTreeFolder2s_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_Browser_Id_Manager_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_CMF_Caching_Policy_Managers_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_CMF_Core_Tools_Permission list           ['Manager', 'ZMSAdministrator']
  _Add_Conte

## Deleted nodes (still in the ZODB until pack)
ZODB never overwrites data: deleting an object writes a new revision of its *parent* without the reference. The child and the older parent revisions stay in `Data.fs` until `pack`. Zope's *Undo* simply re-commits the old parent revision.

`deleted_nodes(parent)` walks the parent's revision history (via historical read-only connections) and lists children that existed in older revisions but are missing now. `open_deleted(info)` returns the old object so that `show()`, `props()` and `scan()` work on it. Restore is done via the Zope *Undo* tab (or writable `set_prop` experiments), not by this notebook.

In [ ]:
import time as _time
# No TimeStamp import needed; history records provide the timestamps.

_hist_conns = []

def deleted_nodes(parent, max_revisions=500):
    """{child_id: info} for children present in older revisions of parent but missing now."""
    oid = parent._p_oid
    current = set(stored_children(parent))
    found = {}
    newer_tid = None
    for rec in db.history(oid, size=max_revisions):  # newest first
        tid = rec['tid']
        hconn = db.open(at=tid)
        try:
            old = hconn.get(oid)
            for k, child in stored_children(old).items():
                if k not in current and k not in found:
                    found[k] = dict(id=k, oid=child._p_oid, class_=describe(child),
                                    last_seen_tid=tid,
                                    last_seen_time=_time.ctime(rec['time']),
                                    deleted_in_tid=newer_tid,
                                    user=rec['user_name'], description=rec['description'])
        finally:
            hconn.close()
        newer_tid = tid
    return found

def show_deleted(parent, **kw):
    res = deleted_nodes(parent, **kw)
    print('== deleted children of', describe(parent), '->', len(res))
    for k, i in sorted(res.items()):
        print('  %-30s oid=%s last seen %s by %r (%s)' % (k, hex(u64(i['oid'])), i['last_seen_time'], i['user'], i['class_']))
    return res

def open_deleted(info):
    """Open the deleted object as it was when last seen (read-only, historical connection)."""
    hconn = db.open(at=info['last_seen_tid'])
    _hist_conns.append(hconn)
    return hconn.get(info['oid'])

def close_deleted():
    while _hist_conns:
        _hist_conns.pop().close()

# deleted = show_deleted(app)               # step 1: list deleted children of a node
# old = open_deleted(deleted['fmltr'])      # step 2: open one
# show(old); props(old)                     # step 3: inspect, continue with walk/scan via stored_children
# close_deleted()

In [35]:
# Show the deleted nodes in the current ZODB application
deleted = show_deleted(app)
old = open_deleted(deleted['fmltr'])
# show(old)
pprint.pprint(props(old))

DB.open() has 7 open connections with a pool_size of 3


== deleted children of Zope oid=0x1 OFS.Application.Application -> 1
  fmltr                          oid=0x180fb last seen Sun Jul  5 16:28:25 2026 by ' admin' (fmltr oid=0x180fb OFS.Folder.Folder)
== fmltr oid=0x180fb OFS.Folder.Folder
  __ac_local_roles__           dict           {'admin': ['Owner']}
  _objects                     tuple          ({'id': 'content', 'meta_type': 'ZMS'}, {'id': 'standard_html', 'meta_type': 'Page Template'},
 {'id': 'zcatalog_index', ... (1905 chars)
  _owner                       tuple          (['acl_users'], 'admin')
  id                           str            'fmltr'
  title                        str            'ZMS - Python-based contentmanagement system for science, technology and medicine'
None


In [16]:
conn.close(); db.close()

---

# Working with ZConsole

## Basic Examples for using zconsole

For a Zope instance:

```Shell
bin/zconsole
```
or for older setups:

```Shell
bin/zopectl debug
```
You should get a Python prompt with the ZODB already opened.

## Access the root object

```Python
app = app() # provided by zconsole
app
```
This returns the Zope application root.

You can inspect its contents:

```Python
app.objectIds()
```
or

```Python
app.objectItems()
```

Example:
```Python
for id in app.objectIds():
    print(id)
```
## Traverse the object hierarchy

Navigate like in the ZMI:

```Python
site = app.unibe
site.objectIds()
```

or dynamically:

```Python
obj = app['unibe']
```

Inspect:

```Python
obj.meta_type
obj.id
obj.title
```
Find ZMS content objects

```Python
for id, obj in site.objectItems():
    print(id, obj.meta_type)
```

Typical ZMS objects expose:

```Python
obj.getTitle()
obj.absolute_url()
obj.meta_type
```

Explore attributes

```Python
dir(obj)
```

or:

```Python
obj.__dict__.keys()
```

## Inspect the underlying persistent object

```Python
from persistent import Persistent
isinstance(obj, Persistent)
```

## Get OID:

```Python
obj._p_oid
```

## Connection:

```Python
obj._p_jar
```

## Walk through a object tree

Useful snippet:

```Python
def walk(obj, level=0):
    print(" " * level + f"{obj.id} ({obj.meta_type})")
    if hasattr(obj, "objectValues"):
        for child in obj.objectValues():
            walk(child, level + 1)
 
walk(app.unibe)
```

## Access ZODB root directly

If you need the actual database root:

```Python
conn = app._p_jar
root = conn.root()
root.keys()
```
Typical output:

```Python
dict_keys(['Application'])
```
Then:

```Python
root['Application']
```

## Useful ZODB inspection

Object size:
```Python
import sys
sys.getsizeof(obj)
```

OID:
```Python
obj._p_oid.hex()
```

State:
```Python
obj._p_changed
```

## Search for objects by meta_type

```Python
def find_by_meta(obj, meta_type):
    if getattr(obj, 'meta_type', None) == meta_type:
        print(obj.absolute_url())
        if hasattr(obj, 'objectValues'):
            for child in obj.objectValues():
                find_by_meta(child, meta_type)
 
find_by_meta(app.unibe, 'ZMSDocument')
```

## Transactions

Read-only exploration is safe. Avoid calling methods that modify objects unless you know what you're doing. If you accidentally change something:

```Python
import transaction
transaction.abort()
```
and never call:
```Python
transaction.commit()
```
unless you intentionally want to persist changes.

Common patterns:

```Python
app = app()
site = app.unibe
 
site.getHome()
site.objectIds()
site.manage_main()
```
which often gives enough entry points to discover the content tree and available methods interactively.
